# Chào mừng đến với tuần học về RAG!!

## Chuyên viên tri thức

### Trợ lý hỏi đáp đóng vai trò là một chuyên viên tri thức
### Dành cho nhân viên của Insurellm, một công ty công nghệ bảo hiểm
### Trợ lý AI cần đưa ra câu trả lời chính xác và giải pháp phải có chi phí thấp.

Dự án này sẽ sử dụng RAG (Retrieval Augmented Generation — Sinh tăng cường truy xuất) để đảm bảo trợ lý hỏi đáp của chúng ta có độ chính xác cao.

Phiên bản triển khai đầu tiên này sẽ sử dụng một kiểu RAG đơn giản, vét cạn.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Ứng dụng kinh doanh của các dự án trong tuần này</h2>
            <span style="color:#181;">RAG có lẽ là kỹ thuật có khả năng ứng dụng tức thì cao nhất trong tất cả những nội dung chúng ta học trong khóa này! Trên thực tế, đã có các sản phẩm thương mại thực hiện chính xác những gì chúng ta xây dựng trong tuần này: truy vấn tinh tế trên các cơ sở dữ liệu thông tin lớn, chẳng hạn như hợp đồng của công ty hoặc thông số kỹ thuật sản phẩm. RAG cung cấp cho bạn một cơ chế có chi phí thấp, nhanh chóng đưa ra thị trường để điều chỉnh LLM phù hợp với lĩnh vực kinh doanh của mình.</span>
        </td>
    </tr>
</table>

In [ ]:
import os
import glob
from dotenv import load_dotenv
from pathlib import Path
import gradio as gr
from openai import OpenAI

In [ ]:
# Thiết lập

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"Khóa API OpenAI tồn tại và bắt đầu bằng {openai_api_key[:8]}")
else:
    print("Chưa thiết lập khóa API OpenAI")

MODEL = "gpt-4.1-nano"
openai = OpenAI()

### Hãy đọc toàn bộ dữ liệu nhân viên vào một từ điển

In [ ]:
knowledge = {}

filenames = glob.glob("knowledge-base/employees/*")

for filename in filenames:
    name = Path(filename).stem.split(' ')[-1]
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

In [ ]:
knowledge

In [ ]:
knowledge["lancaster"]

In [ ]:
filenames = glob.glob("knowledge-base/products/*")

for filename in filenames:
    name = Path(filename).stem
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

In [ ]:
knowledge.keys()

In [ ]:
SYSTEM_PREFIX = """
Bạn đại diện cho Insurellm, công ty công nghệ bảo hiểm.
Bạn là chuyên gia trả lời các câu hỏi về Insurellm, nhân viên và sản phẩm của công ty.
Bạn được cung cấp ngữ cảnh bổ sung có thể liên quan đến câu hỏi của người dùng.
Hãy trả lời ngắn gọn và chính xác. Nếu không biết câu trả lời, hãy nói rõ điều đó.

Ngữ cảnh liên quan:
"""

In [ ]:
def get_relevant_context_simple(message):
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    relevant_context = []
    for word in words:
        if word in knowledge:
            relevant_context.append(knowledge[word])
    return relevant_context          

## Nhưng đây là một cách viết đậm chất Python hơn:

In [ ]:
def get_relevant_context(message):
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    return [knowledge[word] for word in words if word in knowledge]   

In [ ]:
get_relevant_context("Lancaster là ai?")

In [ ]:
get_relevant_context("Lancaster là ai và carllm là gì?")

In [ ]:
def additional_context(message):
    relevant_context = get_relevant_context(message)
    if not relevant_context:
        result = "Không có ngữ cảnh bổ sung nào liên quan đến câu hỏi của người dùng."
    else:
        result = "Ngữ cảnh bổ sung sau đây có thể hữu ích khi trả lời câu hỏi của người dùng:\n\n"
        result += "\n\n".join(relevant_context)
    return result

In [ ]:
print(additional_context("Alex Lancaster là ai?"))

In [ ]:
def chat(message, history):
    system_message = SYSTEM_PREFIX + additional_context(message)
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

## Bây giờ chúng ta sẽ đưa ứng dụng này lên Gradio bằng giao diện Chat

Một cách nhanh chóng và dễ dàng để tạo nguyên mẫu trò chuyện với LLM

In [ ]:
view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)